# Active-model pricing audit

## tl;dr

This notebook executes the repository-wide pricing audit used by the accompanying technical report. The conclusions are populated from executed outputs below; the HTML report is the reader-facing synthesis.

## Context & Methods

The audit uses two complementary definitions as of **18 July 2026 BST / 17 July 2026 23:30 UTC**:

- **Catalog-active model:** `status` is `Available` or `Limited Access`.
- **Production-active route:** a provider-model mapping has `is_active_gateway = true`, its effective window is open, and its capability status is `active` or `deranked_*`.

### Key Assumptions

Pricing coverage means at least one rule is effective at the audit timestamp. Historical and future rules are retained but do not satisfy current coverage. Provider price spreads are anomaly candidates, not automatic errors, because region, quantization, service tier, and promotional terms can legitimately differ.

In [1]:
from pathlib import Path
import json
import sys

current = Path.cwd().resolve()
repo_root = next(candidate for candidate in [current, *current.parents] if (candidate / 'packages/data/catalog/src/data').exists())
audit_dir = repo_root / 'apps/web/scripts/importer/reports/pricing-audit-2026-07-18'
sys.path.insert(0, str(audit_dir))
from pricing_audit import build_audit, write_outputs

audit = build_audit(repo_root)
write_outputs(audit, repo_root)
audit['summary']

{'audit_at': '2026-07-17T23:30:00Z',
 'model_files': 1203,
 'catalog_active_models': 703,
 'available_models': 701,
 'limited_access_models': 2,
 'active_models_with_current_pricing': 417,
 'active_model_pricing_coverage_rate': 0.5931721194879089,
 'active_models_with_gateway_route': 333,
 'active_gateway_models_with_current_pricing': 333,
 'active_gateway_model_pricing_coverage_rate': 1.0,
 'provider_model_routes': 956,
 'provider_model_routes_status_active': 919,
 'provider_model_routes_status_deranked': 37,
 'priced_provider_model_routes': 934,
 'route_pricing_gap_count': 22,
 'route_pricing_gap_count_after_capability_aliases': 11,
 'alternate_pricing_key_gap_count': 11,
 'route_pricing_coverage_rate': 0.9769874476987448,
 'route_pricing_coverage_rate_after_capability_aliases': 0.9884937238493724,
 'pricing_files': 1353,
 'current_pricing_files': 1277,
 'pricing_rules': 4863,
 'current_pricing_rules': 4372,
 'current_free_rules': 33,
 'pricing_providers': 63,
 'current_pricing_provi

## Data

### 1. Coverage classes for catalog-active models

In [2]:
audit['coverage_classes']

[{'coverage_class': 'active_route_and_priced', 'model_count': 333},
 {'coverage_class': 'no_active_route_or_current_price', 'model_count': 286},
 {'coverage_class': 'priced_without_active_route', 'model_count': 84}]

### 2. Active provider capability gaps

In [3]:
{
    'gap_count': len(audit['route_gaps']),
    'by_provider': audit['gap_by_provider'],
    'sample': audit['route_gaps'][:10],
}

{'gap_count': 22,
 'by_provider': [{'provider_id': 'openai', 'missing_route_count': 11},
  {'provider_id': 'avian', 'missing_route_count': 8},
  {'provider_id': 'ambient', 'missing_route_count': 3}],
 'sample': [{'provider_id': 'ambient',
   'model_id': 'moonshotai/kimi-k2.7-code',
   'capability_id': 'text.generate',
   'capability_status': 'active',
   'provider_api_model_id': 'ambient:moonshotai/kimi-k2.7-code',
   'provider_model_slug': 'moonshotai/kimi-k2.7-code',
   'internal_model_id': 'moonshotai/kimi-k2.7-code',
   'capability_params': [],
   'pricing_key': 'ambient:moonshotai/kimi-k2.7-code:text.generate',
   'source_path': 'packages/data/catalog/src/data/api_providers/ambient/models.json',
   'gap_type': 'missing_file',
   'alternate_current_pricing_keys': []},
  {'provider_id': 'ambient',
   'model_id': 'z-ai/glm-5.2',
   'capability_id': 'text.generate',
   'capability_status': 'active',
   'provider_api_model_id': 'ambient:z-ai/glm-5.2',
   'provider_model_slug': 'z-ai/gl

## Results

### 3. Provider coverage and provenance

In [4]:
sorted(
    audit['provider_coverage'],
    key=lambda row: (-row['missing_active_route_count'], -row['active_route_count'], row['provider_id'])
)[:20]

[{'provider_id': 'openai',
  'active_route_count': 118,
  'priced_active_route_count': 107,
  'missing_active_route_count': 11,
  'unresolved_missing_active_route_count': 0,
  'route_pricing_coverage_rate': 0.9067796610169492,
  'route_pricing_coverage_rate_after_capability_aliases': 1.0,
  'pricing_entry_count': 128,
  'current_pricing_entry_count': 128,
  'current_rule_count': 693,
  'has_pricing_source_url': True,
  'pricing_source_url': 'https://developers.openai.com/api/docs/pricing'},
 {'provider_id': 'avian',
  'active_route_count': 8,
  'priced_active_route_count': 0,
  'missing_active_route_count': 8,
  'unresolved_missing_active_route_count': 8,
  'route_pricing_coverage_rate': 0.0,
  'route_pricing_coverage_rate_after_capability_aliases': 0.0,
  'pricing_entry_count': 0,
  'current_pricing_entry_count': 0,
  'current_rule_count': 0,
  'has_pricing_source_url': False,
  'pricing_source_url': None},
 {'provider_id': 'ambient',
  'active_route_count': 3,
  'priced_active_route_

### 4. Rule-level anomaly checks

In [5]:
{
    'duplicate_keys': audit['duplicate_pricing_keys'],
    'conflicting_overlaps': audit['overlapping_rules'][:20],
    'cached_read_more_expensive': audit['cached_read_anomalies'][:20],
    'batch_more_expensive': audit['batch_anomalies'][:20],
    'top_cross_provider_spreads': audit['top_cross_provider_spreads'][:20],
}

{'duplicate_keys': [],
 'conflicting_overlaps': [{'pricing_key': 'anthropic:anthropic/claude-3-opus:text.generate',
   'meter': 'output_text_tokens',
   'pricing_plan': 'batch',
   'priority': 100,
   'left_price': 2.0,
   'left_from': None,
   'left_to': None,
   'right_price': 37.5,
   'right_from': None,
   'right_to': None,
   'source_path': 'packages/data/catalog/src/data/pricing/anthropic/anthropic-claude-3-opus/text.generate/pricing.json',
   'catalog_model_id': 'anthropic/claude-3-opus',
   'model_status': 'Retired'},
  {'pricing_key': 'anthropic-aws:anthropic/claude-3-opus:text.generate',
   'meter': 'output_text_tokens',
   'pricing_plan': 'batch',
   'priority': 100,
   'left_price': 2.0,
   'left_from': None,
   'left_to': None,
   'right_price': 37.5,
   'right_from': None,
   'right_to': None,
   'source_path': 'packages/data/catalog/src/data/pricing/anthropic-aws/anthropic-claude-3-opus/text.generate/pricing.json',
   'catalog_model_id': 'anthropic/claude-3-opus',
   'mo

## Takeaways

The executed summary and detail outputs above are the canonical calculation record. Interpretive prioritization, external-source spot checks, limitations, and recommended remediation are documented in the accompanying HTML report.